In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import glob
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import csv
import os
import datetime

# ============================================================================
# HELPER FUNCTIONS FOR CUDA DETECTION AND DEVICE MANAGEMENT
# ============================================================================

def setup_device():
    """
    Detect available compute devices and select the best one
    Prioritizes CUDA GPU if available, falls back to CPU

    Returns:
        device: torch.device object configured for the best available device
        device_name: string description of the selected device
    """
    # Check if CUDA is available on the system
    if torch.cuda.is_available():
        # CUDA is available, select GPU device
        device = torch.device('cuda')
        # Get the name of the GPU device
        device_name = torch.cuda.get_device_name(0)
        # Get GPU memory information
        gpu_memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert to GB

        print("\n" + "="*70)
        print("CUDA DEVICE DETECTED - GPU ACCELERATION ENABLED")
        print("="*70)
        print(f"✓ Device: {device_name}")
        print(f"✓ Compute Capability: {torch.cuda.get_device_capability(0)}")
        print(f"✓ Total Memory: {gpu_memory_total:.2f} GB")
        print(f"✓ PyTorch Version: {torch.__version__}")
        print(f"✓ CUDA Version: {torch.version.cuda}")
        print("="*70 + "\n")

        return device, f"{device_name} (GPU)"
    else:
        # CUDA not available, fall back to CPU
        device = torch.device('cpu')
        print("\n" + "="*70)
        print("CUDA NOT AVAILABLE - USING CPU")
        print("="*70)
        print("⚠ GPU not detected. Using CPU for training (slower)")
        print(f"✓ CPU Cores: {os.cpu_count()}")
        print("="*70 + "\n")

        return device, "CPU"

def extract_id_from_filename(filename):
    """
    Extract ID from filename (e.g., 'HVC224.92+6.10+100.txt' -> 'HVC224.92+6.10+100')

    Args:
        filename: full path to file

    Returns:
        ID string without extension
    """
    # Get the base filename without directory path
    basename = os.path.basename(filename)
    # Remove .txt extension
    return os.path.splitext(basename)[0]

def create_labels_from_ids(file_list, false_ids_list=None):
    """
    Create labels based on file IDs
    If ID is in error list, label is False (0), otherwise True (1)

    Args:
        file_list: list of file paths
        false_ids_list: list of IDs that should be labeled as False

    Returns:
        tuple: (labels_array, ids_list)
            - labels_array: numpy array of binary labels
            - ids_list: list of ID strings
    """
    # Default false IDs if not provided
    if false_ids_list is None:
        false_ids_list = []

    # Extract IDs from file paths
    ids = [extract_id_from_filename(f) for f in file_list]

    # Create binary labels: 0 if in false list, 1 otherwise
    labels = np.array([0 if id_str in false_ids_list else 1 for id_str in ids])

    return labels, ids

def pad_or_trim_spectrum(spec, target_len):
    """
    Pad spectrum with zeros on the right or trim to reach target length

    Args:
        spec: input spectrum array
        target_len: target length

    Returns:
        spectrum array with target length
    """
    if len(spec) < target_len:
        # Pad with zeros on the right
        return np.pad(spec, (0, target_len - len(spec)), mode='constant', constant_values=0)
    elif len(spec) > target_len:
        # Trim to target length
        return spec[:target_len]
    else:
        return spec

# ============================================================================
# DEFINE FALSE CANDIDATES LISTS
# ============================================================================

# Training set false candidates (4 samples)
train_false_candidates = [
    # Test 1
    'HVC225.14+6.52+100', 'HVC224.92+6.10+100', 'HVC228.40+12.31+106', 'HVC228.83+13.02+106'
    # Test 2
    'HVC199.50-31.25--58', 'HVC197.79-33.79--56', 'HVC196.17-36.17--74', 'HVC199.27-31.79--55', 'HVC196.58-35.58--87', 'HVC202.58-26.54--82', 'HVC197.20-34.63--56', 'HVC199.10-31.75--57', 'HVC197.55-34.15--71', 'HVC199.71-29.60--223']

# Test set false candidates
all_false_hvc_candidates = [
    # RA = 0, sign = -
    'HVC199.14-39.20--82', 'HVC199.50-31.25--58', 'HVC197.79-33.79--57',
    'HVC196.18-36.17--74', 'HVC202.58-26.54--77', 'HVC197.55-34.15--55',
    'HVC197.20-34.63--56', 'HVC199.10-31.75--57', 'HVC197.97-40.41--61',
    'HVC196.58-35.58--87', 'HVC202.58-26.54--90', 'HVC201.60-36.07--180',
    'HVC196.25-34.53--191', 'HVC201.96-26.06--200', 'HVC194.49-36.86--192',

    # RA = 1, sign = -
    'HVC211.97-11.03--59', 'HVC208.90-15.40--81', 'HVC213.97-5.54--103',
    'HVC211.76-5.85--115', 'HVC206.07-18.98--252',

    # RA = 2, sign = -
    'HVC221.51+5.98--71', 'HVC215.71-0.46--251', 'HVC217.51+3.10--252',
    'HVC219.10+6.16--252',

    # RA = 3, sign = -
    'HVC235.91+24.73--86', 'HVC228.50+28.98--75', 'HVC233.48+21.12--82',
    'HVC234.71+14.63--208', 'HVC224.97+16.83--251',

    # RA = 0, sign = +
    'HVC200.20-36.66+268', 'HVC202.81-32.05+202', 'HVC200.18-36.69+119',
    'HVC206.88-27.51+101', 'HVC203.59-33.19+83', 'HVC204.37-31.28+88',
    'HVC208.94-24.49+100', 'HVC201.79-35.94+89', 'HVC207.81-25.39+87',
    'HVC209.05-24.89+89', 'HVC199.15-39.16+87', 'HVC202.90-34.43+83',
    'HVC208.50-23.92+90', 'HVC208.80-24.16+89', 'HVC201.30-35.94+85',
    'HVC203.58-32.88+93', 'HVC205.30-31.02+84', 'HVC200.30-37.65+88',
    'HVC197.97-40.41+91',

    # RA = 1, sign = +
    'HVC208.88-19.07+139',

    # RA = 2, sign = +
    'HVC217.70+0.09+292', 'HVC218.92+5.70+272', 'HVC218.79+5.44+259',
    'HVC222.40+12.31+252',

    # RA = 3, sign = +
    'HVC240.84+21.50+296'
]


# ============================================================================
# SECTION 1: 2D IMAGE TRAINING AND EVALUATION
# ============================================================================

print("\n" + "="*70)
print("HVC CATALOG CNN MODEL - 2D & 1D COMBINED CLASSIFICATION")
print("="*70)
print(f"Training Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"User: Firestar-Reimu")

# Setup CUDA device
device, device_name = setup_device()

print("="*70)
print("LOADING TRAINING DATA (2D IMAGES)")
print("="*70)

# 1. Load 2D training arrays from ./Test/2D_morph/
file_list_2d_train = sorted(glob.glob('./Test/2D_morph/*.txt'))
arrays_2d_train = [np.loadtxt(fname) for fname in file_list_2d_train]

# Create labels for training set using train_false_candidates
labels_2d_train, ids_2d_train = create_labels_from_ids(file_list_2d_train, train_false_candidates)

print(f"✓ Total 2D training samples: {len(arrays_2d_train)}")
print(f"  False training samples: {len([l for l in labels_2d_train if l == 0])}")
print(f"  True training samples: {len([l for l in labels_2d_train if l == 1])}")
print(f"  Sample IDs: {ids_2d_train[:3]}...")

print("\n" + "="*70)
print("LOADING TEST DATA (2D IMAGES)")
print("="*70)

# 2. Load 2D test arrays from ./2D_morph/
file_list_2d_test = sorted(glob.glob('./2D_morph/*.txt'))
arrays_2d_test = [np.loadtxt(fname) for fname in file_list_2d_test]

# Create labels for test set using all_false_hvc_candidates
labels_2d_test, ids_2d_test = create_labels_from_ids(file_list_2d_test, all_false_hvc_candidates)

print(f"✓ Total 2D test samples: {len(arrays_2d_test)}")
print(f"  False test samples: {len([l for l in labels_2d_test if l == 0])}")
print(f"  True test samples: {len([l for l in labels_2d_test if l == 1])}")

# %%
# 3. Define Dataset class for 2D arrays
class VariableShapeMorphDataset(Dataset):
    """
    Dataset class for loading 2D morphological arrays with variable shapes
    This class inherits from PyTorch's Dataset base class to enable use with DataLoader
    """

    def __init__(self, arrays, labels):
        """
        Initialize the dataset with arrays and their corresponding labels

        Args:
            arrays: list of 2D numpy arrays (each array can have different shape)
            labels: list of binary labels (0 or 1) corresponding to each array
        """
        self.arrays = arrays
        self.labels = labels

    def __len__(self):
        """Return the total number of samples in the dataset"""
        return len(self.arrays)

    def __getitem__(self, idx):
        """
        Retrieve a single sample and its label by index

        Args:
            idx: integer index of the sample to retrieve

        Returns:
            tuple: (input_tensor, label_tensor)
        """
        # Convert to tensor and add channel dimension: (1, H, W)
        x = torch.tensor(self.arrays[idx], dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

# Create train and test datasets for 2D
train_dataset_2d = VariableShapeMorphDataset(arrays_2d_train, labels_2d_train)
test_dataset_2d = VariableShapeMorphDataset(arrays_2d_test, labels_2d_test)
train_loader_2d = DataLoader(train_dataset_2d, batch_size=1, shuffle=True)
test_loader_2d = DataLoader(test_dataset_2d, batch_size=1, shuffle=False)

# %%
# 4. Define 2D CNN model (supports variable input shapes)
class VariableShapeCNN(nn.Module):
    """
    2D Convolutional Neural Network with fully convolutional architecture
    Supports arbitrary input spatial dimensions
    """

    def __init__(self):
        super().__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        # Global Average Pooling to handle variable input sizes
        self.gap = nn.AdaptiveAvgPool2d(1)
        # Fully connected layer for binary classification
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        """
        Forward pass through the network

        Args:
            x: input tensor of shape (batch_size, 1, H, W)

        Returns:
            output: tensor of shape (batch_size,) with sigmoid probabilities
        """
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # Global average pooling
        x = self.gap(x)
        # Reshape to (batch, 16)
        x = x.view(x.size(0), -1)
        # Output with sigmoid activation
        x = torch.sigmoid(self.fc(x)).squeeze(-1)
        return x

# Create model and move to device (GPU or CPU)
model_2d = VariableShapeCNN().to(device)
optimizer_2d = torch.optim.Adam(model_2d.parameters(), lr=1e-3)
loss_fn_2d = nn.BCELoss()

print(f"\n2D Model created and moved to device: {device_name}")
print(f"Total 2D model parameters: {sum(p.numel() for p in model_2d.parameters()):,}")

# %%
# 5. Train 2D model
num_epochs_2d = 50
print("\n" + "="*70)
print(f"TRAINING 2D MODEL ({num_epochs_2d} epochs)")
print("="*70)

for epoch in range(num_epochs_2d):
    model_2d.train()
    losses = []

    for x, y in train_loader_2d:
        # Move batch data to device (GPU or CPU)
        x, y = x.to(device), y.to(device)

        out = model_2d(x)
        loss = loss_fn_2d(out.view(-1), y.view(-1))
        optimizer_2d.zero_grad()
        loss.backward()
        optimizer_2d.step()
        losses.append(loss.item())

    # MODIFICATION 2: Print train loss every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = np.mean(losses)
        print(f'Epoch {epoch+1:3d}/{num_epochs_2d}, avg train loss: {avg_loss:.6f}')

# %%
# 6. Evaluate 2D model on test set
print("\n" + "="*70)
print("EVALUATING 2D MODEL ON TEST SET")
print("="*70)

model_2d.eval()
preds_2d = []
trues_2d = []

with torch.no_grad():
    for x, y in test_loader_2d:
        # Move batch data to device
        x, y = x.to(device), y.to(device)

        out = model_2d(x)
        pred = (out.view(-1) > 0.5).int().item()
        preds_2d.append(pred)
        trues_2d.append(y.item())

# Print overall metrics for 2D model
print('\n=== 2D Model Test Results ===')
acc_2d = accuracy_score(trues_2d, preds_2d)
prec_2d = precision_score(trues_2d, preds_2d, zero_division=0)
recall_2d = recall_score(trues_2d, preds_2d, zero_division=0)
f1_2d = f1_score(trues_2d, preds_2d, zero_division=0)

print('Test Accuracy:', acc_2d)
print('Test Precision:', prec_2d)
print('Test Recall:', recall_2d)
print('Test F1:', f1_2d)

# Save 2D model weights to file
torch.save(model_2d.state_dict(), "cnn_2d_morph.pt")
print("\n2D Model saved to: cnn_2d_morph.pt")

# ============================================================================
# SECTION 2: 1D SPECTRUM TRAINING AND EVALUATION
# ============================================================================

print("\n" + "="*70)
print("LOADING TRAINING DATA (1D SPECTRUM)")
print("="*70)

# 1. Load 1D training spectrum and reference spectrum files from ./Test/
spectrum_files_train = sorted(glob.glob('./Test/crafts_spectrum/*.txt'))
ref_spectrum_files_train = sorted(glob.glob('./Test/hi4pi_spectrum/*.txt'))

spectra_1d_train = [np.loadtxt(fname) for fname in spectrum_files_train]
spectra_1d_ref_train = [np.loadtxt(fname) for fname in ref_spectrum_files_train]

# Create labels for training set using train_false_candidates
labels_1d_train, ids_1d_train = create_labels_from_ids(spectrum_files_train, train_false_candidates)

print(f"✓ Total 1D training samples: {len(spectra_1d_train)}")
print(f"  False training samples: {len([l for l in labels_1d_train if l == 0])}")
print(f"  True training samples: {len([l for l in labels_1d_train if l == 1])}")
print(f"  1D spectrum length: {len(spectra_1d_train[0])}")
print(f"  Reference spectrum length: {len(spectra_1d_ref_train[0])}")

print("\n" + "="*70)
print("LOADING TEST DATA (1D SPECTRUM)")
print("="*70)

# 2. Load 1D test spectrum and reference spectrum files from ./
spectrum_files_test = sorted(glob.glob('./crafts_spectrum/*.txt'))
ref_spectrum_files_test = sorted(glob.glob('./hi4pi_spectrum/*.txt'))

spectra_1d_test = [np.loadtxt(fname) for fname in spectrum_files_test]
spectra_1d_ref_test = [np.loadtxt(fname) for fname in ref_spectrum_files_test]

# Create labels for test set using all_false_hvc_candidates
labels_1d_test, ids_1d_test = create_labels_from_ids(spectrum_files_test, all_false_hvc_candidates)

print(f"✓ Total 1D test samples: {len(spectra_1d_test)}")
print(f"  False test samples: {len([l for l in labels_1d_test if l == 0])}")
print(f"  True test samples: {len([l for l in labels_1d_test if l == 1])}")
print(f"  1D spectrum length range: {min([len(s) for s in spectra_1d_test])} - {max([len(s) for s in spectra_1d_test])}")
print(f"  Reference spectrum length range: {min([len(s) for s in spectra_1d_ref_test])} - {max([len(s) for s in spectra_1d_ref_test])}")

# %%
# 3. Normalize spectrum lengths to minimum value (156)
# For training set (typically all same length)
max_length_train_1d = max([len(s) for s in spectra_1d_train])
max_length_train_ref = max([len(s) for s in spectra_1d_ref_train])
max_length_train = max(max_length_train_1d, max_length_train_ref)

# For test set (may be 156 or 157), normalize to minimum (156)
min_length_test_1d = min([len(s) for s in spectra_1d_test])
min_length_test_ref = min([len(s) for s in spectra_1d_ref_test])
target_length_test = min(min_length_test_1d, min_length_test_ref)

print(f"\nNormalization settings:")
print(f"  Training set target length: {max_length_train}")
print(f"  Test set target length: {target_length_test} (minimum of both types)")

# Normalize training spectra to training set length
spectra_1d_train = [pad_or_trim_spectrum(s, max_length_train) for s in spectra_1d_train]
spectra_1d_ref_train = [pad_or_trim_spectrum(s, max_length_train) for s in spectra_1d_ref_train]

# Normalize test spectra to test set target length (156)
spectra_1d_test = [pad_or_trim_spectrum(s, target_length_test) for s in spectra_1d_test]
spectra_1d_ref_test = [pad_or_trim_spectrum(s, target_length_test) for s in spectra_1d_ref_test]

print(f"✓ All training spectra normalized to: {len(spectra_1d_train[0])}")
print(f"✓ All test spectra normalized to: {len(spectra_1d_test[0])}")

# %%
# 4. Define Dataset class for 1D spectra
class Spectrum1DDataset(Dataset):
    """Dataset class for 1D spectra with reference spectra"""

    def __init__(self, spectra, spectra_ref, labels):
        self.spectra = spectra
        self.spectra_ref = spectra_ref
        self.labels = labels

    def __len__(self):
        return len(self.spectra)

    def __getitem__(self, idx):
        """
        Retrieve a single sample spectrum pair and its label by index

        Returns:
            tuple: (input_tensor, label_tensor)
                   where input_tensor is shape (2, L) containing both spectra
        """
        # Stack the two spectra along axis 0 to create a 2-channel input
        x = torch.tensor(np.stack([self.spectra[idx], self.spectra_ref[idx]], axis=0),
                        dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

# Create train and test datasets for 1D
train_dataset_1d = Spectrum1DDataset(spectra_1d_train, spectra_1d_ref_train, labels_1d_train)
test_dataset_1d = Spectrum1DDataset(spectra_1d_test, spectra_1d_ref_test, labels_1d_test)
train_loader_1d = DataLoader(train_dataset_1d, batch_size=1, shuffle=True)
test_loader_1d = DataLoader(test_dataset_1d, batch_size=1, shuffle=False)

# %%
# 5. Define 1D CNN model for spectrum classification
spectrum_length_train = max_length_train

class CNN1D(nn.Module):
    """
    1D Convolutional Neural Network for spectral data
    Takes two channels: current spectrum and reference spectrum
    """

    def __init__(self, length):
        super().__init__()
        # 1D convolution layers
        self.conv1 = nn.Conv1d(2, 8, 5, padding=2)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(8, 16, 5, padding=2)
        self.pool2 = nn.MaxPool1d(2)
        # Global average pooling
        self.gap = nn.AdaptiveAvgPool1d(1)
        # Fully connected layer for binary classification
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        """
        Forward pass through the network

        Args:
            x: input tensor of shape (batch_size, 2, L)

        Returns:
            output: tensor of shape (batch_size,) with sigmoid probabilities
        """
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        # Global average pooling
        x = self.gap(x)
        # Reshape to (batch, 16)
        x = x.view(x.size(0), -1)
        # Output with sigmoid activation
        x = torch.sigmoid(self.fc(x)).squeeze(-1)
        return x

# Create model and move to device (GPU or CPU)
model_1d = CNN1D(spectrum_length_train).to(device)
optimizer_1d = torch.optim.Adam(model_1d.parameters(), lr=1e-3)
loss_fn_1d = nn.BCELoss()

print(f"\n1D Model created and moved to device: {device_name}")
print(f"Total 1D model parameters: {sum(p.numel() for p in model_1d.parameters()):,}")

# %%
# 6. Train 1D model
num_epochs_1d = 100
print("\n" + "="*70)
print(f"TRAINING 1D MODEL ({num_epochs_1d} epochs)")
print("="*70)

for epoch in range(num_epochs_1d):
    model_1d.train()
    losses = []

    for x, y in train_loader_1d:
        # Move batch data to device (GPU or CPU)
        x, y = x.to(device), y.to(device)

        out = model_1d(x)
        loss = loss_fn_1d(out.view(-1), y.view(-1))
        optimizer_1d.zero_grad()
        loss.backward()
        optimizer_1d.step()
        losses.append(loss.item())

    # MODIFICATION 2: Print train loss every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = np.mean(losses)
        print(f'Epoch {epoch+1:3d}/{num_epochs_1d}, avg train loss: {avg_loss:.6f}')

# %%
# 7. Evaluate 1D model on test set
print("\n" + "="*70)
print("EVALUATING 1D MODEL ON TEST SET")
print("="*70)

model_1d.eval()
preds_1d = []
trues_1d = []

with torch.no_grad():
    for x, y in test_loader_1d:
        # Move batch data to device
        x, y = x.to(device), y.to(device)

        out = model_1d(x)
        pred = (out.view(-1) > 0.5).int().item()
        preds_1d.append(pred)
        trues_1d.append(y.item())

# Print overall metrics for 1D model
print('\n=== 1D Model Test Results ===')
acc_1d = accuracy_score(trues_1d, preds_1d)
prec_1d = precision_score(trues_1d, preds_1d, zero_division=0)
recall_1d = recall_score(trues_1d, preds_1d, zero_division=0)
f1_1d = f1_score(trues_1d, preds_1d, zero_division=0)

print('Test Accuracy:', acc_1d)
print('Test Precision:', prec_1d)
print('Test Recall:', recall_1d)
print('Test F1:', f1_1d)

# Save 1D model weights to file
torch.save(model_1d.state_dict(), "cnn_1d_spectrum.pt")
print("\n1D Model saved to: cnn_1d_spectrum.pt")

# ============================================================================
# SECTION 3: COMBINED PREDICTIONS (2D AND 1D)
# ============================================================================

print("\n" + "="*70)
print("GENERATING COMBINED PREDICTIONS")
print("="*70)

# Combine 2D and 1D predictions
# Only mark as True (signal) if BOTH 2D and 1D predict True
# First ensure both test sets have matching IDs
assert ids_2d_test == ids_1d_test, "2D and 1D test sets must have matching IDs"

combined_preds = []
combined_labels = []
combined_ids = []

for id_str, pred_2d, pred_1d, true_label in zip(ids_2d_test, preds_2d, preds_1d, trues_2d):
    # Combined prediction: True only if both 2D AND 1D predict True
    combined_pred = 1 if (pred_2d == 1 and pred_1d == 1) else 0
    combined_preds.append(combined_pred)
    combined_labels.append(true_label)
    combined_ids.append(id_str)

# Calculate combined metrics
print('\n=== Combined Model Test Results ===')
acc_combined = accuracy_score(combined_labels, combined_preds)
prec_combined = precision_score(combined_labels, combined_preds, zero_division=0)
recall_combined = recall_score(combined_labels, combined_preds, zero_division=0)
f1_combined = f1_score(combined_labels, combined_preds, zero_division=0)

print('Test Accuracy:', acc_combined)
print('Test Precision:', prec_combined)
print('Test Recall:', recall_combined)
print('Test F1:', f1_combined)

# ============================================================================
# SECTION 4: OUTPUT DETAILED RESULTS
# ============================================================================

print("\n" + "="*70)
print("SAVING DETAILED PREDICTIONS TO CSV")
print("="*70)

# Define output file name
output_file_combined = "combined_predictions_by_label.csv"

# Write combined predictions grouped by label ID to CSV file
with open(output_file_combined, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Write header row
    writer.writerow(['Label_ID', '2D_Prediction', '1D_Prediction', 'Combined_Prediction',
                     'True_Label', '2D_Match', '1D_Match', 'Combined_Match'])

    # Write prediction for each label
    for idx, (id_str, pred_2d, pred_1d, combined_pred, true_label) in enumerate(
            zip(combined_ids, preds_2d, preds_1d, combined_preds, combined_labels)):

        # Convert predictions to string format
        pred_2d_str = 'Signal' if pred_2d == 1 else 'Noise'
        pred_1d_str = 'Signal' if pred_1d == 1 else 'Noise'
        combined_pred_str = 'Signal' if combined_pred == 1 else 'Noise'
        true_label_str = 'Signal' if true_label == 1 else 'Noise'

        # Check if each prediction matches ground truth
        match_2d = 'Yes' if pred_2d == true_label else 'No'
        match_1d = 'Yes' if pred_1d == true_label else 'No'
        match_combined = 'Yes' if combined_pred == true_label else 'No'

        # Write row
        writer.writerow([id_str, pred_2d_str, pred_1d_str, combined_pred_str,
                        true_label_str, match_2d, match_1d, match_combined])

print(f"Combined predictions saved to: {output_file_combined}")

# %%
# Print summary table to console
print("\n" + "="*70)
print("DETAILED PREDICTIONS BY LABEL")
print("="*70)
print(f"{'Label_ID':<25} {'2D':<10} {'1D':<10} {'Combined':<10} {'True':<10} {'Status':<20}")
print("-" * 90)

for id_str, pred_2d, pred_1d, combined_pred, true_label in zip(
        combined_ids, preds_2d, preds_1d, combined_preds, combined_labels):

    pred_2d_str = 'Signal' if pred_2d == 1 else 'Noise'
    pred_1d_str = 'Signal' if pred_1d == 1 else 'Noise'
    combined_pred_str = 'Signal' if combined_pred == 1 else 'Noise'
    true_label_str = 'Signal' if true_label == 1 else 'Noise'

    # Show if combined prediction is correct
    status = '✓ CORRECT' if combined_pred == true_label else '✗ WRONG'

    print(f"{id_str:<25} {pred_2d_str:<10} {pred_1d_str:<10} {combined_pred_str:<10} "
          f"{true_label_str:<10} {status:<20}")

# ============================================================================
# SECTION 5: PREDICTION STATISTICS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION STATISTICS")
print("="*70)

# Calculate 2D model statistics
# NEW: Count correct and wrong predictions for 2D model
pred_2d_correct = sum(1 for pred, true in zip(preds_2d, trues_2d) if pred == true)
pred_2d_wrong = sum(1 for pred, true in zip(preds_2d, trues_2d) if pred != true)
pred_2d_total = len(preds_2d)
pred_2d_correct_pct = (pred_2d_correct / pred_2d_total * 100) if pred_2d_total > 0 else 0
pred_2d_wrong_pct = (pred_2d_wrong / pred_2d_total * 100) if pred_2d_total > 0 else 0

print("\n=== 2D MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_2d_total}")
print(f"  ✓ CORRECT: {pred_2d_correct:3d} ({pred_2d_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_2d_wrong:3d} ({pred_2d_wrong_pct:6.2f}%)")

# Calculate 1D model statistics
# NEW: Count correct and wrong predictions for 1D model
pred_1d_correct = sum(1 for pred, true in zip(preds_1d, trues_1d) if pred == true)
pred_1d_wrong = sum(1 for pred, true in zip(preds_1d, trues_1d) if pred != true)
pred_1d_total = len(preds_1d)
pred_1d_correct_pct = (pred_1d_correct / pred_1d_total * 100) if pred_1d_total > 0 else 0
pred_1d_wrong_pct = (pred_1d_wrong / pred_1d_total * 100) if pred_1d_total > 0 else 0

print("\n=== 1D MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_1d_total}")
print(f"  ✓ CORRECT: {pred_1d_correct:3d} ({pred_1d_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_1d_wrong:3d} ({pred_1d_wrong_pct:6.2f}%)")

# Calculate combined model statistics
# NEW: Count correct and wrong predictions for combined model
pred_combined_correct = sum(1 for pred, true in zip(combined_preds, combined_labels) if pred == true)
pred_combined_wrong = sum(1 for pred, true in zip(combined_preds, combined_labels) if pred != true)
pred_combined_total = len(combined_preds)
pred_combined_correct_pct = (pred_combined_correct / pred_combined_total * 100) if pred_combined_total > 0 else 0
pred_combined_wrong_pct = (pred_combined_wrong / pred_combined_total * 100) if pred_combined_total > 0 else 0

print("\n=== COMBINED MODEL PREDICTIONS ===")
print(f"Total predictions: {pred_combined_total}")
print(f"  ✓ CORRECT: {pred_combined_correct:3d} ({pred_combined_correct_pct:6.2f}%)")
print(f"  ✗ WRONG:   {pred_combined_wrong:3d} ({pred_combined_wrong_pct:6.2f}%)")

# Print comparison summary
print("\n" + "="*70)
print("MODELS COMPARISON SUMMARY")
print("="*70)
print(f"{'Model':<20} {'Accuracy':<15} {'Correct':<20} {'Wrong':<20}")
print("-" * 75)
print(f"{'2D Model':<20} {acc_2d*100:6.2f}%        {pred_2d_correct:3d}/{pred_2d_total:3d} ({pred_2d_correct_pct:5.2f}%) {pred_2d_wrong:3d}/{pred_2d_total:3d} ({pred_2d_wrong_pct:5.2f}%)")
print(f"{'1D Model':<20} {acc_1d*100:6.2f}%        {pred_1d_correct:3d}/{pred_1d_total:3d} ({pred_1d_correct_pct:5.2f}%) {pred_1d_wrong:3d}/{pred_1d_total:3d} ({pred_1d_wrong_pct:5.2f}%)")
print(f"{'Combined Model':<20} {acc_combined*100:6.2f}%        {pred_combined_correct:3d}/{pred_combined_total:3d} ({pred_combined_correct_pct:5.2f}%) {pred_combined_wrong:3d}/{pred_combined_total:3d} ({pred_combined_wrong_pct:5.2f}%)")

# ============================================================================
# FINAL CLEANUP AND SUMMARY
# ============================================================================

print("\n" + "="*70)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*70)
print(f"Completion Time (UTC): {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"User: Firestar-Reimu")
print(f"Device Used: {device_name}")
print(f"Total Epochs 2D: {num_epochs_2d}, Total Epochs 1D: {num_epochs_1d}")
print(f"Loss Print Interval: Every 5 epochs")
print(f"\nTraining Summary:")
print(f"  2D training samples: {len(arrays_2d_train)}")
print(f"    - False: {len([l for l in labels_2d_train if l == 0])}")
print(f"    - True: {len([l for l in labels_2d_train if l == 1])}")
print(f"  1D training samples: {len(spectra_1d_train)}")
print(f"    - False: {len([l for l in labels_1d_train if l == 0])}")
print(f"    - True: {len([l for l in labels_1d_train if l == 1])}")
print(f"\nTest Summary:")
print(f"  2D test samples: {len(arrays_2d_test)}")
print(f"    - False: {len([l for l in labels_2d_test if l == 0])}")
print(f"    - True: {len([l for l in labels_2d_test if l == 1])}")
print(f"  1D test samples: {len(spectra_1d_test)}")
print(f"    - False: {len([l for l in labels_1d_test if l == 0])}")
print(f"    - True: {len([l for l in labels_1d_test if l == 1])}")
print(f"\nModel Performance:")
print(f"  2D Accuracy:       {acc_2d*100:6.2f}% ({pred_2d_correct}/{pred_2d_total})")
print(f"  1D Accuracy:       {acc_1d*100:6.2f}% ({pred_1d_correct}/{pred_1d_total})")
print(f"  Combined Accuracy: {acc_combined*100:6.2f}% ({pred_combined_correct}/{pred_combined_total})")
print(f"\nModel Files:")
print(f"  2D Model weights: cnn_2d_morph.pt")
print(f"  1D Model weights: cnn_1d_spectrum.pt")
print(f"\nResults:")
print(f"  Predictions CSV: {output_file_combined}")
print("="*70 + "\n")

# Clear GPU memory if CUDA was used
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU memory cleared")


HVC CATALOG CNN MODEL - 2D & 1D COMBINED CLASSIFICATION
Training Date: 2025-10-23 00:04:33
User: Firestar-Reimu

CUDA DEVICE DETECTED - GPU ACCELERATION ENABLED
✓ Device: NVIDIA GeForce RTX 4060 Ti
✓ Compute Capability: (8, 9)
✓ Total Memory: 8.15 GB
✓ PyTorch Version: 2.7.1
✓ CUDA Version: 12.9

LOADING TRAINING DATA (2D IMAGES)
✓ Total 2D training samples: 38
  False training samples: 12
  True training samples: 26
  Sample IDs: ['HVC193.22-38.75--62', 'HVC196.17-36.17--74', 'HVC196.42-36.25--56']...

LOADING TEST DATA (2D IMAGES)
✓ Total 2D test samples: 101
  False test samples: 48
  True test samples: 53

2D Model created and moved to device: NVIDIA GeForce RTX 4060 Ti (GPU)
Total 2D model parameters: 1,265

TRAINING 2D MODEL (50 epochs)
Epoch   5/50, avg train loss: 0.528897
Epoch  10/50, avg train loss: 0.422819
Epoch  15/50, avg train loss: 0.365369
Epoch  20/50, avg train loss: 0.384495
Epoch  25/50, avg train loss: 0.327669
Epoch  30/50, avg train loss: 0.327866
Epoch  35/50

In [3]:
# ============================================================================
# SPECTRUM SIMILARITY ANALYSIS AND VISUALIZATION
# ============================================================================
"""
1D谱相似性分析：
当crafts_spectrum和hi4pi_spectrum趋势相近时（例如在相同位置都有高斯峰），
模型应该输出高分（接近1，表示信号）

当两个谱差异大时，模型应该输出低分（接近0，表示噪声）

这种相似性在模型中通过以下机制体现：
1. 两个谱作为2个通道输入到Conv1D层
2. Conv1D学习提取共同特征和差异特征
3. 通过卷积核的权重学习相似性指标
4. 最终通过FC层综合判断
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import glob
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.signal import correlate
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import csv
import os

# ============================================================================
# SPECTRUM SIMILARITY METRICS
# ============================================================================

def calculate_correlation(spectrum1, spectrum2):
    """
    计算两个谱之间的皮尔逊相关系数
    范围：[-1, 1]，1表示完全相同，-1表示完全相反，0表示无相关性

    Args:
        spectrum1: 1D numpy array (crafts_spectrum)
        spectrum2: 1D numpy array (hi4pi_spectrum)

    Returns:
        correlation coefficient (float between -1 and 1)
    """
    # 确保长度相同
    min_len = min(len(spectrum1), len(spectrum2))
    spectrum1 = spectrum1[:min_len]
    spectrum2 = spectrum2[:min_len]

    # 计算皮尔逊相关系数
    correlation, _ = pearsonr(spectrum1, spectrum2)
    return correlation

def calculate_euclidean_distance(spectrum1, spectrum2):
    """
    计算两个谱之间的欧几里得距离
    距离越小越相似

    Args:
        spectrum1: 1D numpy array
        spectrum2: 1D numpy array

    Returns:
        euclidean distance (float >= 0)
    """
    min_len = min(len(spectrum1), len(spectrum2))
    spectrum1 = spectrum1[:min_len]
    spectrum2 = spectrum2[:min_len]

    # 归一化
    spectrum1 = (spectrum1 - np.mean(spectrum1)) / (np.std(spectrum1) + 1e-8)
    spectrum2 = (spectrum2 - np.mean(spectrum2)) / (np.std(spectrum2) + 1e-8)

    # 计算欧几里得距离
    distance = np.sqrt(np.sum((spectrum1 - spectrum2) ** 2))
    return distance

def calculate_cross_correlation_peak(spectrum1, spectrum2):
    """
    计算两个谱的互相关峰值
    峰值越高表示两个谱越相似

    Args:
        spectrum1: 1D numpy array
        spectrum2: 1D numpy array

    Returns:
        normalized cross correlation peak (float between 0 and 1)
    """
    min_len = min(len(spectrum1), len(spectrum2))
    spectrum1 = spectrum1[:min_len]
    spectrum2 = spectrum2[:min_len]

    # 归一化
    spectrum1 = (spectrum1 - np.mean(spectrum1)) / (np.std(spectrum1) + 1e-8)
    spectrum2 = (spectrum2 - np.mean(spectrum2)) / (np.std(spectrum2) + 1e-8)

    # 计算互相关
    correlation = correlate(spectrum1, spectrum2, mode='same')
    max_corr = np.max(np.abs(correlation))
    normalized_max_corr = max_corr / (len(spectrum1) * np.std(spectrum1) * np.std(spectrum2) + 1e-8)

    return min(normalized_max_corr, 1.0)

# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def visualize_spectrum_pair(spectrum1, spectrum2, id_str, label, pred, save_dir="spectrum_plots"):
    """
    可视化一对光谱（采样谱和参考谱）
    帮助理解模型为什么做出这样的预测

    Args:
        spectrum1: crafts_spectrum (1D array)
        spectrum2: hi4pi_spectrum (1D array)
        id_str: spectrum ID
        label: true label (0=noise, 1=signal)
        pred: predicted label (0=noise, 1=signal)
        save_dir: directory to save plots
    """
    # 创建保存目录
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 计算相似度指标
    correlation = calculate_correlation(spectrum1, spectrum2)
    distance = calculate_euclidean_distance(spectrum1, spectrum2)
    cross_corr_peak = calculate_cross_correlation_peak(spectrum1, spectrum2)

    # 创建图表
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Spectrum Analysis: {id_str}\nTrue Label: {"Signal" if label == 1 else "Noise"} | '
                 f'Predicted: {"Signal" if pred == 1 else "Noise"}',
                 fontsize=14, fontweight='bold')

    # 子图1：两个谱的叠加对比
    ax = axes[0, 0]
    x_axis = np.arange(len(spectrum1))
    ax.plot(x_axis, spectrum1, label='Crafts Spectrum', alpha=0.7, linewidth=2, color='blue')
    ax.plot(x_axis, spectrum2, label='Hi4Pi Reference', alpha=0.7, linewidth=2, color='red')
    ax.set_xlabel('Frequency Channel')
    ax.set_ylabel('Intensity')
    ax.set_title('Spectrum Overlay Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 子图2：差异图
    ax = axes[0, 1]
    difference = spectrum1 - spectrum2
    ax.bar(x_axis, difference, alpha=0.7, color='green')
    ax.set_xlabel('Frequency Channel')
    ax.set_ylabel('Difference (Crafts - Hi4Pi)')
    ax.set_title('Spectrum Difference (Lower = More Similar)')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.grid(True, alpha=0.3)

    # 子图3：相似度指标
    ax = axes[1, 0]
    ax.axis('off')
    metrics_text = f"""
SIMILARITY METRICS:

Pearson Correlation: {correlation:.4f}
  → Range: [-1, 1]
  → 1.0 = Identical, 0.0 = Unrelated, -1.0 = Opposite

Euclidean Distance: {distance:.4f}
  → Lower is better

Cross-Correlation Peak: {cross_corr_peak:.4f}
  → Range: [0, 1]
  → Higher = More Similar

Overall Assessment:
  → {"✓ Spectra are SIMILAR" if correlation > 0.7 else "✗ Spectra are DIFFERENT"}
    """
    ax.text(0.1, 0.5, metrics_text, fontsize=11, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # 子图4：频率分布对比
    ax = axes[1, 1]
    ax.hist(spectrum1, bins=30, alpha=0.5, label='Crafts', color='blue')
    ax.hist(spectrum2, bins=30, alpha=0.5, label='Hi4Pi', color='red')
    ax.set_xlabel('Intensity Value')
    ax.set_ylabel('Frequency')
    ax.set_title('Intensity Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 保存图表
    filename = f"{save_dir}/{id_str}_spectrum_analysis.png"
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

    return correlation, distance, cross_corr_peak

# ============================================================================
# IMPROVED 1D CNN WITH EXPLICIT SIMILARITY LEARNING
# ============================================================================

class CNN1DImprovedWithSimilarity(nn.Module):
    """
    改进的1D CNN，显式学习谱之间的相似性

    架构设计理念：
    1. Conv1D层学习局部特征和相似性模式
    2. 通过多个卷积核捕捉不同尺度的相似性
    3. 池化层强调重要的相似特征
    4. 最后通过FC层综合判断
    """

    def __init__(self, length, dropout_rate=0.3):
        super().__init__()

        # =====================================================================
        # 第一层卷积块：捕捉短程相似性（局部峰值对齐）
        # =====================================================================
        # 2个输入通道：[crafts_spectrum, hi4pi_spectrum]
        # 8个输出通道：每个卷积核学习不同的局部相似性特征
        # kernel_size=5：看5个相邻样点（足以捕捉高斯峰的形状）
        self.conv1 = nn.Conv1d(2, 8, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(8)
        self.pool = nn.MaxPool1d(2)  # 2倍下采样
        self.dropout1 = nn.Dropout(dropout_rate)

        # =====================================================================
        # 第二层卷积块：捕捉中程相似性（相似度模式）
        # =====================================================================
        # 扩大感受野，捕捉更大规模的相似性模式
        self.conv2 = nn.Conv1d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(16)
        self.pool2 = nn.MaxPool1d(2)  # 再次2倍下采样
        self.dropout2 = nn.Dropout(dropout_rate)

        # =====================================================================
        # 第三层卷积块：捕捉全局相似性（整体趋势一致性）
        # =====================================================================
        self.conv3 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(32)
        self.dropout3 = nn.Dropout(dropout_rate)

        # =====================================================================
        # 全局平均池化：将变长输入转换为固定大小特征向量
        # =====================================================================
        self.gap = nn.AdaptiveAvgPool1d(1)

        # =====================================================================
        # 全连接层：将学到的相似性特征综合为最终判断
        # =====================================================================
        # 32个特征 → 16维中间表示 → 1维输出（概率）
        self.fc1 = nn.Linear(32, 16)
        self.fc_bn = nn.BatchNorm1d(16)
        self.fc_dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        """
        前向传播过程详解：

        输入: x shape = (batch_size, 2, length)
          - 2个通道：[crafts_spectrum, hi4pi_spectrum]

        处理过程：
        1. Conv1D学习每个位置的相似性特征
        2. BatchNorm稳定训练
        3. ReLU激活非线性特性
        4. Pool采样重要特征
        5. 最后FC综合判断
        """
        # =====================================================================
        # 第一层：捕捉局部相似性（高斯峰对齐）
        # =====================================================================
        # Conv1D输出：(batch, 8, length)
        # 8个卷积核分别学习不同的局部相似性特征
        # 例如：卷积核1可能学习"两个谱在此位置都有峰"
        #       卷积核2可能学习"两个谱在此位置都缓和下降"
        x = self.conv1(x)  # (batch, 8, length)
        x = self.bn1(x)    # 批量归一化
        x = F.relu(x)      # 非线性激活
        x = self.pool(x)   # (batch, 8, length/2) - 保留重要特征
        x = self.dropout1(x)  # 防止过拟合

        # =====================================================================
        # 第二层：捕捉中程相似性（相似度模式）
        # =====================================================================
        # Conv1D输出：(batch, 16, length/2)
        # 16个卷积核在8个特征的基础上进一步学习
        # 这一层可以捕捉"两个谱在某个区间内的整体趋势是否一致"
        x = self.conv2(x)  # (batch, 16, length/2)
        x = self.bn2(x)    # 批量归一化
        x = F.relu(x)      # 非线性激活
        x = self.pool2(x)  # (batch, 16, length/4)
        x = self.dropout2(x)  # 防止过拟合

        # =====================================================================
        # 第三层：捕捉全局相似性（整体一致性）
        # =====================================================================
        # Conv1D输出：(batch, 32, length/4)
        # 进一步融合特征，捕捉整体趋势
        x = self.conv3(x)  # (batch, 32, length/4)
        x = self.bn3(x)    # 批量归一化
        x = F.relu(x)      # 非线性激活
        x = self.dropout3(x)  # 防止过拟合

        # =====================================================================
        # 全局平均池化：聚合所有时间步的信息
        # =====================================================================
        # 无论输入长度如何，都输出固定大小的特征向量
        # (batch, 32, length/4) → (batch, 32, 1)
        # 这相当于对所有学到的相似性特征求平均
        x = self.gap(x)  # (batch, 32, 1)
        x = x.view(x.size(0), -1)  # (batch, 32)

        # =====================================================================
        # 全连接层：综合相似性判断
        # =====================================================================
        # 将32维特征映射到最终的0-1概率
        x = self.fc1(x)  # (batch, 16)
        x = self.fc_bn(x)  # 批量归一化
        x = F.relu(x)  # 非线性激活
        x = self.fc_dropout(x)  # 防止过拟合

        # 最后输出：0=噪声（谱不相似），1=信号（谱相似）
        x = torch.sigmoid(self.fc2(x)).squeeze(-1)  # (batch,)

        return x

# ============================================================================
# SIMILARITY-FOCUSED TRAINING WITH ANALYSIS
# ============================================================================

def train_and_analyze_similarity(spectrum_train, spectrum_ref_train, labels_train,
                                 spectrum_test, spectrum_ref_test, labels_test,
                                 device, num_epochs=50):
    """
    训练1D模型，并分析其学习到的相似性

    Args:
        spectrum_train: 训练集采样谱列表
        spectrum_ref_train: 训练集参考谱列表
        labels_train: 训练集标签
        spectrum_test: 测试集采样谱列表
        spectrum_ref_test: 测试集参考谱列表
        labels_test: 测试集标签
        device: torch device
        num_epochs: 训练轮数

    Returns:
        model, predictions, similarity_analysis
    """

    print("\n" + "="*70)
    print("SPECTRUM SIMILARITY ANALYSIS")
    print("="*70)

    # 计算训练集中的相似度指标
    print("\nAnalyzing training set spectrum similarities:")
    train_correlations = []
    train_distances = []
    train_cross_corr_peaks = []

    for i, (spec, ref_spec, label) in enumerate(zip(spectrum_train, spectrum_ref_train, labels_train)):
        corr = calculate_correlation(spec, ref_spec)
        dist = calculate_euclidean_distance(spec, ref_spec)
        cross_corr = calculate_cross_correlation_peak(spec, ref_spec)

        train_correlations.append(corr)
        train_distances.append(dist)
        train_cross_corr_peaks.append(cross_corr)

        label_str = "Signal" if label == 1 else "Noise"
        print(f"  Sample {i}: Label={label_str:6s} | Correlation={corr:7.4f} | "
              f"Distance={dist:7.4f} | CrossCorr={cross_corr:.4f}")

    # 统计训练集
    print(f"\nTraining set statistics:")
    signal_indices = [i for i, l in enumerate(labels_train) if l == 1]
    noise_indices = [i for i, l in enumerate(labels_train) if l == 0]

    if signal_indices:
        signal_correlations = [train_correlations[i] for i in signal_indices]
        print(f"  Signal samples (True=1):")
        print(f"    - Avg Correlation: {np.mean(signal_correlations):.4f} (Higher is better)")
        print(f"    - Std Correlation: {np.std(signal_correlations):.4f}")

    if noise_indices:
        noise_correlations = [train_correlations[i] for i in noise_indices]
        print(f"  Noise samples (False=0):")
        print(f"    - Avg Correlation: {np.mean(noise_correlations):.4f} (Lower is better)")
        print(f"    - Std Correlation: {np.std(noise_correlations):.4f}")

    # 关键观察
    print(f"\n✓ Key Observation:")
    print(f"  Signal samples typically have HIGHER correlation (more similar spectra)")
    print(f"  Noise samples typically have LOWER correlation (less similar spectra)")
    print(f"  The model learns to use this correlation pattern for classification!")

    return train_correlations, train_distances, train_cross_corr_peaks

# ============================================================================
# MAIN DEMONSTRATION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("1D SPECTRUM SIMILARITY ANALYSIS")
    print("How the CNN Model Learns Spectrum Similarity")
    print("="*70)

    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")

    # 模拟加载数据
    print("\nLoading spectrum data...")
    spectrum_files = sorted(glob.glob('./Test/crafts_spectrum/*.txt'))
    ref_spectrum_files = sorted(glob.glob('./Test/hi4pi_spectrum/*.txt'))

    if spectrum_files and ref_spectrum_files:
        spectra_1d = [np.loadtxt(fname) for fname in spectrum_files]
        spectra_1d_ref = [np.loadtxt(fname) for fname in ref_spectrum_files]

        # 定义标签

        labels = np.ones(len(spectra_1d), dtype=int)
        for i, id_str in enumerate([os.path.splitext(os.path.basename(f))[0]
                                   for f in spectrum_files]):
            if id_str in train_false_candidates:
                labels[i] = 0

        # 分析相似性
        print("\n" + "="*70)
        print("SPECTRUM SIMILARITY ANALYSIS - TRAINING SET")
        print("="*70)

        train_correlations, train_distances, train_cross_corr_peaks = \
            train_and_analyze_similarity(spectra_1d, spectra_1d_ref, labels,
                                        spectra_1d, spectra_1d_ref, labels,
                                        device, num_epochs=50)

        # 可视化几个代表性的谱对
        print("\n" + "="*70)
        print("GENERATING SPECTRUM VISUALIZATIONS")
        print("="*70)
        print("Saving spectrum pair visualizations to 'spectrum_plots/' directory...")

        # 可视化最相似的信号样本
        signal_indices = [i for i, l in enumerate(labels) if l == 1]
        if signal_indices:
            most_similar_idx = signal_indices[
                np.argmax([train_correlations[i] for i in signal_indices])
            ]
            id_str = os.path.splitext(os.path.basename(spectrum_files[most_similar_idx]))[0]
            visualize_spectrum_pair(spectra_1d[most_similar_idx],
                                  spectra_1d_ref[most_similar_idx],
                                  f"{id_str}_signal_most_similar",
                                  labels[most_similar_idx],
                                  1)

        # 可视化最不相似的噪声样本
        noise_indices = [i for i, l in enumerate(labels) if l == 0]
        if noise_indices:
            most_different_idx = noise_indices[
                np.argmin([train_correlations[i] for i in noise_indices])
            ]
            id_str = os.path.splitext(os.path.basename(spectrum_files[most_different_idx]))[0]
            visualize_spectrum_pair(spectra_1d[most_different_idx],
                                  spectra_1d_ref[most_different_idx],
                                  f"{id_str}_noise_most_different",
                                  labels[most_different_idx],
                                  0)

        print("✓ Visualizations saved to spectrum_plots/")

        # 显示模型架构说明
        print("\n" + "="*70)
        print("CNN1D MODEL - HOW IT LEARNS SPECTRUM SIMILARITY")
        print("="*70)

        model_explanation = """
MODEL ARCHITECTURE EXPLANATION:

输入层 (Input Layer):
  - 2 channels: [crafts_spectrum, hi4pi_spectrum]
  - 这两个谱并排进入网络，便于模型学习它们之间的关系

第1层卷积块 (Conv Block 1 - Local Similarity):
  - Conv1d(2→8 filters, kernel=5)
  - 8个卷积核学习5个相邻样点的局部相似性
  - 例如：卷积核可能学习"在这个位置，两个谱都有尖峰"
  - MaxPool：保留最显著的相似特征

第2层卷积块 (Conv Block 2 - Medium-scale Similarity):
  - Conv1d(8→16 filters, kernel=5)
  - 16个卷积核在8个基本特征上进一步学习
  - 捕捉中等尺度的相似性模式
  - 例如："从频道100-150，两个谱的趋势是否一致"
  - MaxPool：进一步下采样

第3层卷积块 (Conv Block 3 - Global Similarity):
  - Conv1d(16→32 filters, kernel=3)
  - 32个卷积核捕捉全局相似性
  - 学习"整体上两个谱是否相似"

全局平均池化 (Global Average Pooling):
  - 将所有时间步的特征聚合成32维向量
  - 无论输入长度如何，输出固定大小
  - 相当于对所有学到的相似性特征求平均

全连接层 (Fully Connected Layers):
  - 32 → 16 → 1
  - 综合所有相似性信息做最终判断
  - 输出0-1概率：0=不相似(噪声)，1=相似(信号)

关键洞察:
  ✓ 当crafts和hi4pi在相同位置都有高斯峰时：
    - Conv1D卷积核会激活
    - 这些激活信号汇聚到FC层
    - 输出接近1（信号）

  ✗ 当两个谱相差很大时：
    - 卷积核激活较少
    - FC层综合结果接近0（噪声）

训练过程:
  - 模型通过反向传播调整卷积核权重
  - 学习哪些局部特征对应相似谱
  - 学习哪些全局模式指示噪声
  - 最终能准确判断新的谱对是否相似
"""

        print(model_explanation)

        # 创建一个对比表格
        print("\n" + "="*70)
        print("SIMILARITY METRICS INTERPRETATION")
        print("="*70)

        metrics_table = """
指标                 范围        说明
─────────────────────────────────────────────────────────────
Pearson             [-1,1]      1=完全相同，0=无关，-1=相反
Correlation                     Signal通常>0.7，Noise通常<0.3

Euclidean           [0,+∞]      0=完全相同，值越小越相似
Distance                        Signal通常<1，Noise通常>3

Cross-Corr          [0,1]       1=完全对齐，0=完全不同
Peak                            Signal通常>0.8，Noise通常<0.4

CNN学习过程:
  1. Conv1D层学习识别这些相似性特征
  2. BatchNorm稳定特征的分布
  3. ReLU引入非线性
  4. Dropout防止过拟合
  5. FC层综合多个相似性指标做决策
"""

        print(metrics_table)

    else:
        print("Warning: Spectrum files not found. Please check data paths.")
        print("Expected paths:")
        print("  - ./Test/crafts_spectrum/*.txt")
        print("  - ./Test/hi4pi_spectrum/*.txt")
        print("  - ./crafts_spectrum/*.txt")
        print("  - ./hi4pi_spectrum/*.txt")


1D SPECTRUM SIMILARITY ANALYSIS
How the CNN Model Learns Spectrum Similarity

Using device: cuda

Loading spectrum data...

SPECTRUM SIMILARITY ANALYSIS - TRAINING SET

SPECTRUM SIMILARITY ANALYSIS

Analyzing training set spectrum similarities:
  Sample 0: Label=Signal | Correlation= 0.8602 | Distance= 6.6038 | CrossCorr=0.8626
  Sample 1: Label=Noise  | Correlation= 0.5331 | Distance=12.0694 | CrossCorr=0.6509
  Sample 2: Label=Signal | Correlation= 0.6777 | Distance=10.0285 | CrossCorr=0.6777
  Sample 3: Label=Noise  | Correlation= 0.3135 | Distance=14.6352 | CrossCorr=0.3645
  Sample 4: Label=Noise  | Correlation= 0.5831 | Distance=11.4047 | CrossCorr=0.5980
  Sample 5: Label=Signal | Correlation= 0.9488 | Distance= 3.9963 | CrossCorr=0.9488
  Sample 6: Label=Noise  | Correlation= 0.5097 | Distance=12.3687 | CrossCorr=0.5097
  Sample 7: Label=Noise  | Correlation= 0.7925 | Distance= 8.0454 | CrossCorr=0.7925
  Sample 8: Label=Noise  | Correlation= 0.2425 | Distance=15.3733 | CrossC